# Module 6: JSON Functions in Databricks

Databricks provides powerful functions for working with semi-structured JSON data:

| Function | Description |
| --- | --- |
| **from_json** | Parses a JSON string into a Spark struct/map using a **defined schema** |
| **parse_json** | Parses a JSON string into a **VARIANT** type (no schema needed) |
| **VARIANT type** | A flexible data type that stores any JSON-compatible value |

**Key difference:**
* `from_json` requires you to know the schema upfront and returns a typed struct
* `parse_json` returns a VARIANT, which is schema-less and more flexible for evolving data

## 1. from_json()

`from_json(json_str, schema)` parses a JSON string column into a **Spark StructType** or **MapType** using an explicit schema definition.

**When to use:**
* You know the JSON structure in advance
* You need typed columns for downstream operations
* You want Spark to validate the JSON against a schema

**Behavior:**
* Returns `null` for fields that don't match the schema
* Supports nested structs and arrays
* The schema can be defined as a DDL string or a StructType object

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

# Define the expected JSON schema
json_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("skills", ArrayType(StringType()), True),
    StructField("address", StructType([
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
    ]), True),
])

# Sample JSON string data
json_data = [
    ('{"id": 1, "name": "Alice", "skills": ["Python", "SQL"], "address": {"city": "Austin", "state": "TX"}}',),
    ('{"id": 2, "name": "Bob", "skills": ["Java", "Scala"], "address": {"city": "Boston", "state": "MA"}}',),
    ('{"id": 3, "name": "Charlie", "skills": ["Spark"], "address": {"city": "Chicago", "state": "IL"}}',),
]

df = spark.createDataFrame(json_data, ["json_str"])

# Parse the JSON string into a struct using the schema
parsed_df = df.select(from_json(col("json_str"), json_schema).alias("parsed"))
display(parsed_df)

# Flatten individual fields from the parsed struct
flattened_df = df.select(
    from_json(col("json_str"), json_schema).alias("p")
).select(
    col("p.id").alias("id"),
    col("p.name").alias("name"),
    col("p.skills").alias("skills"),
    col("p.address.city").alias("city"),
    col("p.address.state").alias("state"),
)
display(flattened_df)

## 2. parse_json()

`parse_json(json_str)` converts a JSON string into a **VARIANT** value. Unlike `from_json`, it does **not** require a schema definition.

**When to use:**
* The JSON structure varies between rows or evolves over time
* You don't want to maintain a schema
* You need to query nested fields without upfront parsing

**Behavior:**
* Returns a VARIANT type that can hold any JSON-compatible value
* Fields are accessed using colon syntax: `variant_col:field_name`
* Supports nested navigation: `variant_col:parent:child`
* Cast to specific types with `::TYPE` syntax

In [0]:
# Sample JSON string data with varying structure
event_data = [
    ('{"user": "alice", "action": "login", "metadata": {"ip": "10.0.0.1", "device": "mobile"}}',),
    ('{"user": "bob", "action": "purchase", "metadata": {"ip": "10.0.0.2", "device": "desktop"}, "amount": 99.99}',),
    ('{"user": "charlie", "action": "logout", "metadata": {"ip": "10.0.0.3", "device": "tablet"}}',),
]

spark.createDataFrame(event_data, ["json_str"]).createOrReplaceTempView("events_raw")

# parse_json converts the JSON string into a VARIANT column
result = spark.sql("""
SELECT 
    json_str,
    parse_json(json_str) AS variant_data
FROM events_raw
""")
display(result)

# Access nested fields from VARIANT using colon syntax
result2 = spark.sql("""
WITH parsed AS (
    SELECT parse_json(json_str) AS v FROM events_raw
)
SELECT 
    v:user       AS user_name,
    v:action     AS action_type,
    v:metadata:ip     AS ip_address,
    v:metadata:device AS device_type,
    v:amount     AS amount
FROM parsed
""")
display(result2)

# Filter rows using VARIANT fields
result3 = spark.sql("""
WITH parsed AS (
    SELECT parse_json(json_str) AS v FROM events_raw
)
SELECT v:user AS user_name, v:action AS action_type
FROM parsed
WHERE v:action::STRING = 'purchase'
""")
display(result3)

## 3. VARIANT Type

`VARIANT` is a Databricks data type designed for semi-structured data. It can store any JSON-compatible value — objects, arrays, strings, numbers, booleans, or null.

**When to use VARIANT columns:**
* Event logs where each event type has a different schema
* API responses with evolving field sets
* Configuration data with flexible keys

**Accessing VARIANT fields:**

| Syntax | Description |
| --- | --- |
| `col:field` | Extract a field as VARIANT |
| `col:field::STRING` | Extract and cast to a specific type |
| `col:parent:child` | Navigate nested objects |
| `col:items[0]` | Access array elements |
| `variant_get(col, '$.field', 'STRING')` | Function-based extraction with path and type |

**Size limits:** 16 MiB per value (default), up to 128 MiB (extended)

**Performance note:** VARIANT is flexible but may be slower than concrete types for highly structured data. Use concrete types when the schema is stable.

In [0]:
# Create sample product data with JSON attributes
product_data = [
    (1, 'Laptop',   '{"brand": "Dell", "specs": {"ram": "16GB", "cpu": "i7"}, "tags": ["electronics", "computers"], "in_stock": true}'),
    (2, 'Phone',    '{"brand": "Apple", "specs": {"ram": "8GB", "cpu": "A16"}, "tags": ["electronics", "mobile"], "in_stock": true}'),
    (3, 'Tablet',   '{"brand": "Samsung", "specs": {"ram": "12GB", "cpu": "Snapdragon"}, "tags": ["electronics", "tablet"], "in_stock": false}'),
    (4, 'Monitor',  '{"brand": "LG", "specs": {"size": "27inch", "resolution": "4K"}, "tags": ["electronics"], "in_stock": true}'),
]

spark.createDataFrame(product_data, ["product_id", "product_name", "json_str"]).createOrReplaceTempView("products_raw")

# Convert JSON string to VARIANT and query nested fields
result = spark.sql("""
WITH parsed AS (
    SELECT product_id, product_name, parse_json(json_str) AS attr
    FROM products_raw
)
SELECT 
    product_id,
    product_name,
    attr:brand              AS brand,
    attr:specs:ram          AS ram,
    attr:specs:cpu          AS cpu,
    attr:specs:size         AS screen_size,
    attr:specs:resolution   AS resolution,
    attr:tags               AS tags,
    attr:in_stock           AS in_stock
FROM parsed
""")
display(result)

# Cast VARIANT fields to concrete types for analysis
result2 = spark.sql("""
WITH parsed AS (
    SELECT product_id, product_name, parse_json(json_str) AS attr
    FROM products_raw
)
SELECT 
    product_id,
    product_name,
    attr:brand::STRING        AS brand,
    attr:specs:ram::STRING    AS ram_size,
    attr:specs:cpu::STRING    AS cpu_type,
    attr:in_stock::BOOLEAN    AS in_stock
FROM parsed
""")
display(result2)

# Filter using VARIANT fields and access array elements
result3 = spark.sql("""
WITH parsed AS (
    SELECT product_id, product_name, parse_json(json_str) AS attr
    FROM products_raw
)
SELECT 
    product_name,
    attr:brand::STRING        AS brand,
    attr:tags[0]::STRING      AS first_tag,
    attr:in_stock::BOOLEAN    AS in_stock
FROM parsed
WHERE attr:in_stock::BOOLEAN = true
ORDER BY product_id
""")
display(result3)

# Use variant_get for function-based extraction with explicit path and type
result4 = spark.sql("""
WITH parsed AS (
    SELECT product_id, product_name, parse_json(json_str) AS attr
    FROM products_raw
)
SELECT 
    product_id,
    product_name,
    variant_get(attr, '$.brand', 'STRING')         AS brand,
    variant_get(attr, '$.specs.cpu', 'STRING')      AS cpu,
    variant_get(attr, '$.specs.ram', 'STRING')      AS ram
FROM parsed
""")
display(result4)

## Summary: from_json vs parse_json vs VARIANT

| Feature | from_json | parse_json | VARIANT column |
| --- | --- | --- | --- |
| **Input** | JSON string | JSON string | Any JSON-compatible value |
| **Output type** | StructType / MapType | VARIANT | VARIANT |
| **Schema required?** | Yes (explicit) | No | No |
| **Field access** | `col.field` (dot syntax) | `col:field` (colon syntax) | `col:field` (colon syntax) |
| **Type safety** | High (schema-validated) | Low (schema-less) | Low (schema-less) |
| **Best for** | Known, stable schemas | Evolving / variable schemas | Storing semi-structured data in tables |
| **Performance** | Faster (typed) | Flexible (untyped) | Flexible but may be slower than concrete types |

**Quick reference:**
```sql
-- from_json: schema-based parsing
SELECT from_json('{"a":1}', 'STRUCT<a:INT>') AS result;

-- parse_json: schema-less VARIANT parsing
SELECT parse_json('{"a":1}') AS result;

-- VARIANT field access with casting
SELECT parse_json('{"a":1}'):a::INT AS result;
```

## Additional PySpark JSON Functions (DataFrame API)

Beyond `from_json` and `parse_json`, PySpark provides these DataFrame-level JSON functions — all usable with `pyspark.sql.functions` without writing SQL:

| Function | Description |
| --- | --- |
| `to_json(col)` | Converts a struct or map column into a JSON **string** |
| `get_json_object(col, path)` | Extracts a single field from a JSON **string** using JSONPath (`$.field`) |
| `json_tuple(col, f1, f2, ...)` | Extracts multiple top-level fields as separate columns (no nesting) |
| `json_array_length(col)` | Returns the number of elements in a JSON array **string** |
| `schema_of_json(col)` | Infers the schema from a JSON string and returns it as a DDL string |

**Key notes:**
* `get_json_object` and `json_tuple` operate on **raw JSON strings**, not parsed structs or VARIANTs
* `json_tuple` is faster than multiple `get_json_object` calls but only supports top-level fields (no nested paths)
* `schema_of_json` can be combined with `from_json` for **dynamic schema inference** — no manual schema definition needed

In [0]:
from pyspark.sql.functions import (
    from_json, to_json, get_json_object, json_tuple, json_array_length,
    schema_of_json, col, struct,
)

# Sample data with JSON strings
pyspark_data = [
    (1, '{"name": "Alice", "age": 30, "skills": ["Python", "SQL"]}'),
    (2, '{"name": "Bob", "age": 25, "skills": ["Java", "Scala"]}'),
    (3, '{"name": "Charlie", "age": 35, "skills": ["Spark", "Airflow"]}'),
]

pyspark_df = spark.createDataFrame(pyspark_data, ["id", "json_str"])

# 1. schema_of_json -- infer DDL schema from a JSON string
inferred = pyspark_df.select(schema_of_json(col("json_str")).alias("inferred_schema")).limit(1)
display(inferred)

# 2. get_json_object -- extract a single field by JSONPath
extracted = pyspark_df.select(
    col("id"),
    get_json_object(col("json_str"), "$.name").alias("name"),
    get_json_object(col("json_str"), "$.age").alias("age"),
    get_json_object(col("json_str"), "$.skills[0]").alias("first_skill"),
)
display(extracted)

# 3. json_tuple -- extract multiple top-level fields at once
# Faster than multiple get_json_object calls, but no nested field support
tuple_result = pyspark_df.select(
    col("id"),
    json_tuple(col("json_str"), "name", "age"),
)
display(tuple_result)

# 4. json_array_length -- count elements in a JSON array field
array_len = pyspark_df.select(
    col("id"),
    json_array_length(get_json_object(col("json_str"), "$.skills")).alias("skill_count"),
)
display(array_len)

# 5. to_json -- convert a struct column back to a JSON string
struct_df = pyspark_df.select(
    col("id"),
    struct(
        get_json_object(col("json_str"), "$.name").alias("name"),
        get_json_object(col("json_str"), "$.age").alias("age"),
    ).alias("person_info"),
)
json_output = struct_df.select(
    col("id"),
    to_json(col("person_info")).alias("json_output"),
)
display(json_output)

# 6. Dynamic parsing -- combine schema_of_json with from_json
# Infer the schema from the first row, then use it to parse all rows
inferred_ddl = pyspark_df.select(schema_of_json(col("json_str"))).limit(1).collect()[0][0]
print(f"Inferred schema DDL: {inferred_ddl}")

dynamic_parsed = pyspark_df.select(
    col("id"),
    from_json(col("json_str"), inferred_ddl).alias("parsed"),
)
display(dynamic_parsed)
